# Notebook 05: Training with Optax

## Why This Matters

Optax is the standard optimizer library for JAX. It is designed the same way as JAX itself: pure functions, composable transforms. An optimizer in Optax is not an object that mutates parameters -- it is a pair of pure functions: `init(params) -> state` and `update(grads, state, params) -> (updates, new_state)`. This design makes it trivial to combine optimizers, add gradient clipping, or wrap with learning rate schedules. Understanding Optax is required to train any non-trivial JAX model.


In [ ]:
%pip install -q jax jaxlib flax optax numpy matplotlib

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import numpy as np
import matplotlib.pyplot as plt
import time
print("Optax version:", optax.__version__)
print("Ready.")

## 1. Optax Optimizer as a Pure Function

The Optax API has two components:
1. `optimizer.init(params)` -- create the optimizer state (e.g., Adam's first and second moment estimates)
2. `optimizer.update(grads, state, params)` -- compute parameter updates given gradients

The optimizer returns **updates** (not new params). You apply them with `optax.apply_updates(params, updates)`.

This separation is philosophically aligned with JAX: state is explicit, updates are pure functions of inputs.


In [ ]:
# Optax optimizer basics
import optax
import jax
import jax.numpy as jnp

# Simple example with Adam
optimizer = optax.adam(learning_rate=1e-3)

# Create dummy params
params = {'W': jnp.ones((4, 4)), 'b': jnp.zeros(4)}

# Step 1: initialize optimizer state
opt_state = optimizer.init(params)
print('Optimizer state type:', type(opt_state))
print('State components:', [type(s).__name__ for s in opt_state])

# Step 2: compute fake gradients
grads = {'W': jnp.ones((4, 4)) * 0.1, 'b': jnp.ones(4) * 0.1}

# Step 3: compute updates
updates, new_opt_state = optimizer.update(grads, opt_state, params)
print('\nUpdates W:', updates['W'][0, :3], '...')

# Step 4: apply updates to params
new_params = optax.apply_updates(params, updates)
print('Param W before:', params['W'][0, :3])
print('Param W after: ', new_params['W'][0, :3])
print('Difference:    ', (new_params['W'] - params['W'])[0, :3])

## 2. Common Optimizers

Optax provides all standard optimizers. Important choices and their tradeoffs:

| Optimizer | Formula | Best for | Notes |
|-----------|---------|---------|-------|
| `sgd` | $\theta \leftarrow \theta - \eta g$ | Convex problems, large batch | Fastest per step |
| `adam` | Adaptive moments | Default for most problems | lr often needs tuning |
| `adamw` | Adam + weight decay | Transformers, modern NNs | Preferred over adam |
| `lion` | Sign gradient update | Large-scale training | Memory efficient |
| `adagrad` | Per-param lr scaling | Sparse features, NLP | lr never resets |


In [ ]:
# Compare optimizers
def minimize_rosenbrock(optimizer, n_steps=500):
    # Rosenbrock: f(x,y) = (1-x)^2 + 100(y - x^2)^2
    def rosenbrock(params):
        x, y = params
        return (1 - x)**2 + 100 * (y - x**2)**2
    
    grad_fn = jax.jit(jax.value_and_grad(rosenbrock))
    params = jnp.array([-1.0, 1.0])  # start far from minimum (1, 1)
    opt_state = optimizer.init(params)
    
    losses = []
    for _ in range(n_steps):
        loss, grads = grad_fn(params)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        losses.append(float(loss))
    return params, losses

optimizers = {
    'SGD lr=0.001':    optax.sgd(1e-3),
    'Adam lr=0.01':    optax.adam(1e-2),
    'AdamW lr=0.01':   optax.adamw(1e-2),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, opt in optimizers.items():
    final_params, losses = minimize_rosenbrock(opt)
    axes[0].semilogy(losses, label=name)
    print(f'{name}: final params = ({final_params[0]:.4f}, {final_params[1]:.4f}), loss = {losses[-1]:.6f}')

axes[0].set_title('Rosenbrock Minimization')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss (log scale)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate schedules
steps = jnp.arange(1000)
cosine_lr = optax.cosine_decay_schedule(init_value=1e-3, decay_steps=1000)
warmup_cosine = optax.warmup_cosine_decay_schedule(init_value=0, peak_value=1e-3, warmup_steps=100, decay_steps=1000)

axes[1].plot(steps, [cosine_lr(s) for s in steps], label='Cosine decay')
axes[1].plot(steps, [warmup_cosine(s) for s in steps], label='Warmup + cosine')
axes[1].set_title('LR Schedules')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Learning rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('optax_optimizers.png', dpi=100)
plt.show()

## 3. Gradient Clipping and Optimizer Chaining

Optax's most powerful feature is **chaining transformations** with `optax.chain()`. You can compose gradient clipping, weight decay, learning rate warmup, and the optimizer update all in one object.

This is the Optax equivalent of PyTorch's `clip_grad_norm_` + `optimizer.step()`, but written functionally.


In [ ]:
# Optax chaining: clip + adam + weight decay
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),   # clip gradients
    optax.scale_by_adam(),             # adam scaling
    optax.add_decayed_weights(1e-4),   # L2 weight decay
    optax.scale_by_learning_rate(1e-3) # apply lr
)

# This is equivalent to:
# optimizer = optax.adamw(1e-3, weight_decay=1e-4) with gradient clipping

# Convenience version:
optimizer_simple = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate=1e-3, weight_decay=1e-4)
)

params = {'W': jnp.ones((8, 8)), 'b': jnp.zeros(8)}
opt_state = optimizer_simple.init(params)

# Large gradient -- will be clipped
large_grads = {'W': jnp.ones((8, 8)) * 100.0, 'b': jnp.ones(8) * 100.0}
updates, _ = optimizer_simple.update(large_grads, opt_state, params)

grad_norm = jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree_util.tree_leaves(large_grads)))
update_norm = jnp.sqrt(sum(jnp.sum(u**2) for u in jax.tree_util.tree_leaves(updates)))
print(f'Gradient norm (before clip): {grad_norm:.2f}')
print(f'Update norm (after clip+adam): {update_norm:.4f}')

## 4. Full Training Loop: MNIST Classifier

Putting it all together: Flax NNX model + Optax optimizer + JIT-compiled training step. This is the canonical JAX training loop pattern.


In [ ]:
from flax import nnx

# Simple MNIST-style MLP
class Classifier(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.l1 = nnx.Linear(784, 256, rngs=rngs)
        self.l2 = nnx.Linear(256, 128, rngs=rngs)
        self.l3 = nnx.Linear(128, 10, rngs=rngs)
    
    def __call__(self, x):
        x = jax.nn.relu(self.l1(x))
        x = jax.nn.relu(self.l2(x))
        return self.l3(x)

# Loss function
def cross_entropy_loss(logits, labels):
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    return -jnp.mean(log_probs[jnp.arange(len(labels)), labels])

# Training step -- pure function over state
@nnx.jit
def train_step(model, optimizer, x, y):
    def loss_fn(model):
        logits = model(x)
        return cross_entropy_loss(logits, y)
    
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)
    return loss

# Initialize
rngs = nnx.Rngs(0)
model = Classifier(rngs)
optimizer = nnx.Optimizer(model, optax.adamw(1e-3))

# Synthetic MNIST-like data
key = jax.random.PRNGKey(42)
N = 5000
X = jax.random.normal(key, (N, 784))
Y = jax.random.randint(key, (N,), 0, 10)

# Training loop
batch_size = 128
n_epochs = 5
losses = []

t0 = time.time()
for epoch in range(n_epochs):
    epoch_losses = []
    for i in range(0, N, batch_size):
        x_batch = X[i:i+batch_size]
        y_batch = Y[i:i+batch_size]
        loss = train_step(model, optimizer, x_batch, y_batch)
        epoch_losses.append(float(loss))
    mean_loss = np.mean(epoch_losses)
    losses.extend(epoch_losses)
    print(f'Epoch {epoch+1}: loss = {mean_loss:.4f}')

print(f'\nTraining time: {time.time()-t0:.2f}s')

# Evaluate
logits = model(X)
preds = jnp.argmax(logits, axis=-1)
acc = jnp.mean(preds == Y)
print(f'Train accuracy: {acc:.4f} (random = 0.1 for 10 classes)')

## Summary

### Key Concepts

| Concept | Optax API | Notes |
|---------|-----------|-------|
| Initialize optimizer | `opt.init(params)` | Returns optimizer state |
| Compute updates | `opt.update(grads, state, params)` | Returns (updates, new_state) |
| Apply updates | `optax.apply_updates(params, updates)` | Adds updates to params |
| Chain transforms | `optax.chain(clip, adam, ...)` | Compose arbitrarily |
| Gradient clipping | `optax.clip_by_global_norm(1.0)` | Apply before optimizer |
| LR schedule | `optax.warmup_cosine_decay_schedule(...)` | Pass as `learning_rate` arg |

**Next**: `06_pmap_and_data_parallelism.ipynb` -- scaling to multiple devices.
